# Day-1 Data Forensics
Runnable version of `docs/round3/day1_protocol.md`. Every cell is CPU-cheap and feeds a
mechanism decision (3DGUT on/off, appearance gate, sky gate, holdout design, partition need).
Record conclusions in a `DATA_NOTES.md` next to the data.

In [ ]:
# Run from the repo root; adjust if the notebook lives elsewhere
import os, sys
REPO = os.path.abspath(".")
assert os.path.isdir(os.path.join(REPO, "original")), "run this notebook from the repo root"
sys.path.insert(0, os.path.join(REPO, "original"))
DATA_ROOT = os.environ.get("DATA_ROOT", "dataset/raw")   # <-- EDIT: where the competition data lives
RUNS = os.environ.get("RUNS_ROOT", "runs")
os.makedirs(RUNS, exist_ok=True)
print("repo:", REPO, "| data:", DATA_ROOT)

## 1. Inventory + camera forensics → 3DGUT decision

In [ ]:
!python original/preflight.py --data_root $DATA_ROOT

In [ ]:
# Camera details per scene (model, k1) straight from COLMAP
from colmap_loader import read_intrinsics_binary
import glob, os
for scene in sorted(glob.glob(os.path.join(DATA_ROOT, "*", "train", "sparse", "0"))):
    name = scene.split(os.sep)[-4]
    try:
        cams = read_intrinsics_binary(os.path.join(scene, "cameras.bin"))
        for cid, c in list(cams.items())[:1]:
            k1 = c.params[3] if c.model in ("SIMPLE_RADIAL", "RADIAL") else 0.0
            print(f"{name:20s} {c.model:15s} {c.width}x{c.height}  k1={k1:+.5f}  -> "
                  + ("--ut (3DGUT)" if abs(k1) > 0 else "antialiased (no --ut)")
                  + ("  [WARP render: large negative k1]" if k1 < -0.05 else ""))
    except Exception as e:
        print(f"{name:20s} !! {e}")

## 2. Pose sanity → trust-or-redo SfM (threshold: median reproj < 1.5 px → TRUST)

In [ ]:
import numpy as np, struct, glob, os, random
from colmap_loader import read_extrinsics_binary, read_intrinsics_binary, qvec2rotmat

def read_points3d_with_ids(path):
    # minimal COLMAP points3D.bin reader that KEEPS point ids (the repo loader drops them)
    xyz = {}
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        for _ in range(n):
            props = struct.unpack("<QdddBBBd", f.read(43))
            xyz[props[0]] = np.array(props[1:4])
            tl = struct.unpack("<Q", f.read(8))[0]
            f.read(8 * tl)
    return xyz

# KNOWN QUIRK (Rounds 1-2): images.bin keypoints can be stored at a DIFFERENT resolution
# than cameras.bin (organisers ran COLMAP at full res: towers 4x, chair 1.5x). We try
# candidate obs-scales and report the best - preflight.py does the same check.
SCALES = [1.0, 1.5, 2.0, 4.0]

for sp in sorted(glob.glob(os.path.join(DATA_ROOT, "*", "train", "sparse", "0"))):
    name = sp.split(os.sep)[-4]
    try:
        imgs = read_extrinsics_binary(os.path.join(sp, "images.bin"))
        cams = read_intrinsics_binary(os.path.join(sp, "cameras.bin"))
        xyz = read_points3d_with_ids(os.path.join(sp, "points3D.bin"))
        proj, obs = [], []
        for im in random.sample(list(imgs.values()), min(20, len(imgs))):
            R, t = qvec2rotmat(im.qvec), im.tvec
            c = cams[im.camera_id]
            if c.model in ("SIMPLE_RADIAL", "RADIAL", "SIMPLE_PINHOLE"):
                fx = fy = c.params[0]; cx, cy = c.params[1], c.params[2]
            else:  # PINHOLE
                fx, fy, cx, cy = c.params[0], c.params[1], c.params[2], c.params[3]
            for xy, pid in zip(im.xys, im.point3D_ids):
                if pid == -1 or pid not in xyz: continue
                X = R @ xyz[pid] + t
                if X[2] <= 0: continue
                proj.append((fx * X[0] / X[2] + cx, fy * X[1] / X[2] + cy)); obs.append(xy)
        proj = np.array(proj); ob = np.array(obs)
        best_s, best_med = None, float("inf")
        for s in SCALES:
            med = float(np.median(np.hypot(*(proj - ob / s).T)))
            if med < best_med: best_s, best_med = s, med
        verdict = "TRUST organiser poses" if best_med < 1.5 else "INVESTIGATE / GLOMAP contingency"
        note = f" [keypoints stored at {best_s}x camera res]" if best_s != 1.0 else ""
        print(f"{name:20s} median reproj {best_med:6.3f} px @obs-scale {best_s} ({len(ob)} obs) -> {verdict}{note}")
    except Exception as e:
        print(f"{name:20s} !! {e}")
print("Distorted models reproject through the pinhole part only; k1 adds a few px at corners - sub-2px medians are fine.")

## 3. Test-pose structure → holdout design (+ scene extent → partition decision)

In [ ]:
import csv, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import glob, os
from colmap_loader import read_extrinsics_binary, qvec2rotmat
for scene_dir in sorted(glob.glob(os.path.join(DATA_ROOT, "*"))):
    name = os.path.basename(scene_dir)
    csvp = os.path.join(scene_dir, "test", "test_poses.csv")
    sp = os.path.join(scene_dir, "train", "sparse", "0")
    if not (os.path.exists(csvp) and os.path.isdir(sp)): continue
    imgs = read_extrinsics_binary(os.path.join(sp, "images.bin"))
    trc = np.array([-qvec2rotmat(i.qvec).T @ i.tvec for i in imgs.values()])
    with open(csvp) as f:
        rows = list(csv.DictReader(f))
    tec = np.array([-qvec2rotmat(np.array([float(r["qw"]), float(r["qx"]), float(r["qy"]), float(r["qz"])])).T
                    @ np.array([float(r["tx"]), float(r["ty"]), float(r["tz"])]) for r in rows])
    ext = (trc.max(0) - trc.min(0))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for a, (i, j) in zip(ax, [(0, 1), (0, 2)]):
        a.scatter(trc[:, i], trc[:, j], s=3, label="train")
        a.scatter(tec[:, i], tec[:, j], s=8, marker="x", label="test")
        a.set_title(f"{name} axes {i}{j}"); a.legend()
    fig.savefig(f"{RUNS}/forensics_{name}_poses.png", dpi=110); plt.close(fig)
    print(f"{name:20s} train {len(trc):5d} test {len(tec):4d}  extent {ext.round(1)}  -> {RUNS}/forensics_{name}_poses.png")
print("Interleaved test inside the capture -> isolated every-k holdout. Separate pass/area -> mimic that.")

## 4. Photometric drift → appearance gate (threshold: systematic inter-pass drift > ~2/255)

In [ ]:
from PIL import Image
import numpy as np, glob, os
for scene_dir in sorted(glob.glob(os.path.join(DATA_ROOT, "*"))):
    imdir = os.path.join(scene_dir, "train", "images")
    if not os.path.isdir(imdir): continue
    files = sorted(glob.glob(os.path.join(imdir, "*")))[::max(1, len(glob.glob(os.path.join(imdir, '*'))) // 60)]
    lumas, wbs = [], []
    for f in files:
        a = np.asarray(Image.open(f).convert("RGB").resize((160, 120)), dtype=np.float32)
        lumas.append(a.mean()); wbs.append(a.reshape(-1, 3).mean(0))
    lumas = np.array(lumas); wbs = np.array(wbs)
    drift = lumas.max() - lumas.min()
    rg = (wbs[:, 0] / wbs[:, 1]); rg_drift = rg.max() - rg.min()
    print(f"{os.path.basename(scene_dir):20s} luma range {drift:6.2f}/255  R/G ratio range {rg_drift:.4f}  -> "
          + ("APPEARANCE GATE OPEN (check per-pass structure!)" if drift > 8 else "skip appearance machinery"))

## 5. Decisions
Fill in and copy into `DATA_NOTES.md`:
- 3DGUT: ___ | Poses: TRUST / redo: ___ | Holdout: every-k / other: ___
- Appearance gate: open/closed ___ | Sky gate: ___ | Partition needed: ___
Then launch the first probe (smallest scene, `configs/recipes/round3_default.args`, 2–5k iters).